# 1. Data Loading and Exploratory Data Analysis (EDA)
We will start by loading the dataset and performing a quick exploration to understand its structure, columns, and missing values.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
file_path = '/content/train (1).csv'
df = pd.read_csv(file_path)

# Display basic information and first few rows
print("Dataset Shape:", df.shape)
print("\n--- Column Information ---")
display(df.info())
print("\n--- First 5 Rows ---")
display(df.head())

Dataset Shape: (909604, 16)

--- Column Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 909604 entries, 0 to 909603
Data columns (total 16 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   timestamp                               909604 non-null  object 
 1   active_power_calculated_by_converter    909604 non-null  float64
 2   active_power_raw                        909604 non-null  float64
 3   ambient_temperature                     909604 non-null  float64
 4   generator_speed                         909604 non-null  float64
 5   generator_winding_temp_max              909604 non-null  float64
 6   grid_power10min_average                 909604 non-null  float64
 7   nc1_inside_temp                         909604 non-null  float64
 8   nacelle_temp                            909604 non-null  float64
 9   reactice_power_calculated_by_converter  909604 non-null  f

None


--- First 5 Rows ---


,timestamp,active_power_calculated_by_converter,active_power_raw,ambient_temperature,generator_speed,generator_winding_temp_max,grid_power10min_average,nc1_inside_temp,nacelle_temp,reactice_power_calculated_by_converter,reactive_power,wind_direction_raw,wind_speed_raw,wind_speed_turbulence,turbine_id,Target
0,2021-02-19 20:18:00,816.636759,834.917206,31.694380,1159.616602,65.954214,917.897085,31.881972,31.504713,141.457644,165.501518,280.864782,7.057000,0.544082,Turbine_108,47.582787
1,2021-04-27 04:55:00,419.107829,421.050873,12.894948,928.747996,59.571319,445.554250,32.423705,32.755770,89.186457,113.835236,299.552460,5.474937,0.469031,Turbine_18,46.070328
2,2021-01-25 06:26:00,1303.530558,1337.566142,16.648388,1201.219775,61.270498,1364.716003,11.446849,18.332985,230.622309,281.452253,84.960106,8.092457,0.622318,Turbine_105,39.989236
3,2021-10-30 03:47:00,61.494872,53.481008,28.388141,769.806122,40.674348,14.324897,34.253204,32.662889,66.211015,75.017531,87.261119,4.071032,0.760719,Turbine_15,46.056587
4,2021-03-15 00:39:00,593.514364,611.659108,31.519527,1046.916768,64.341763,599.020172,32.405586,31.466387,137.163938,160.202421,313.724818,6.357943,0.346068,Turbine_01,54.346095


# 2. Comprehensive Data Cleaning & Preprocessing
We will define a robust cleaning process including:
- Identifying and handling missing values (imputation)
- Handling duplicate rows
- Automatically detecting categorical and numerical columns
- Encoding categorical variables
- Feature scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Drop duplicate rows if any
df = df.drop_duplicates()

# 2. Extract datetime features from timestamp to avoid explosive dimensionality
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['hour'] = df['timestamp'].dt.hour

# Drop original timestamp
X = df.drop(columns=['timestamp', 'Target'])
y = df['Target']

# 3. Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical Columns ({len(num_cols)}):", num_cols)
print(f"Categorical Columns ({len(cat_cols)}):", cat_cols)

# 4. Create Preprocessing Pipelines
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Let's use a subset of 50000 samples for model training & validation to ensure fast execution
sample_size = 50000
if len(X_train) > sample_size:
    print(f"Sampling {sample_size} rows for training to speed up modeling...")
    X_train_sub = X_train.sample(n=sample_size, random_state=42)
    y_train_sub = y_train.loc[X_train_sub.index]
    X_val_sub = X_val.sample(n=int(sample_size * 0.25), random_state=42)
    y_val_sub = y_val.loc[X_val_sub.index]
else:
    X_train_sub, y_train_sub, X_val_sub, y_val_sub = X_train, y_train, X_val, y_val

# Fit and transform
X_train_prep = preprocessor.fit_transform(X_train_sub)
X_val_prep = preprocessor.transform(X_val_sub)
y_train = y_train_sub.values
y_val = y_val_sub.values

print("\nPreprocessing Complete.")
print("Preprocessed Training Shape:", X_train_prep.shape)

Numerical Columns (13): ['active_power_calculated_by_converter', 'active_power_raw', 'ambient_temperature', 'generator_speed', 'generator_winding_temp_max', 'grid_power10min_average', 'nc1_inside_temp', 'nacelle_temp', 'reactice_power_calculated_by_converter', 'reactive_power', 'wind_direction_raw', 'wind_speed_raw', 'wind_speed_turbulence']
Categorical Columns (1): ['turbine_id']
Sampling 50000 rows for training to speed up modeling...

Preprocessing Complete.
Preprocessed Training Shape: (50000, 29)


# 3. Regression Modeling
We will now train multiple regression algorithms (Linear Regression, Ridge, Lasso, Decision Tree, Random Forest, Gradient Boosting, and Support Vector Regression) and compare their performance using metrics like R-squared and Root Mean Squared Error (RMSE).

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Initialize models (Excluding expensive SVR for speed over 50k rows)
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'Decision Tree': DecisionTreeRegressor(max_depth=7, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=50, random_state=42)
}

# Evaluate models
results = []

for name, model in models.items():
    print(f"Training {name}...")
    # Train the model
    model.fit(X_train_prep, y_train)

    # Predict
    y_pred = model.predict(X_val_prep)

    # Metrics
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)

    results.append({
        'Model': name,
        'RMSE': rmse,
        'R2': r2
    })

# Convert results to DataFrame and display
results_df = pd.DataFrame(results).sort_values(by='R2', ascending=False)
display(results_df)

Training Linear Regression...
Training Ridge...
Training Lasso...
Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...


,Model,RMSE,R2
4,Random Forest,1.439326,0.699980
5,Gradient Boosting,1.698974,0.581972
3,Decision Tree,1.702976,0.580000
1,Ridge,1.743439,0.559804
0,Linear Regression,1.743447,0.559800
2,Lasso,2.019769,0.409206


## 4. Manual Preprocessing and Data Cleaning (No Pipelines)
We will manually clean, impute, encode, and scale the dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Deduplicate
df_clean = df.drop_duplicates().copy()

# 2. Re-extract datetime features
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
df_clean['year'] = df_clean['timestamp'].dt.year
df_clean['month'] = df_clean['timestamp'].dt.month
df_clean['day'] = df_clean['timestamp'].dt.day
df_clean['hour'] = df_clean['timestamp'].dt.hour

# Drop original timestamp and split features and target
X_manual = df_clean.drop(columns=['timestamp', 'Target'])
y_manual = df_clean['Target']

# 3. Manual Imputation (Median for numeric, Mode for categorical)
numeric_cols = X_manual.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_manual.select_dtypes(include=['object', 'category']).columns.tolist()

for col in numeric_cols:
    median_val = X_manual[col].median()
    X_manual[col] = X_manual[col].fillna(median_val)

for col in categorical_cols:
    mode_val = X_manual[col].mode()[0]
    X_manual[col] = X_manual[col].fillna(mode_val)

# 4. Manual One-Hot Encoding
X_encoded = pd.get_dummies(X_manual, columns=categorical_cols, drop_first=True, dtype=float)

# 5. Train/Validation Split
X_train_man, X_val_man, y_train_man, y_val_man = train_test_split(X_encoded, y_manual, test_size=0.2, random_state=42)

# Subset to 50,000 samples for efficient training
sample_size = 50000
if len(X_train_man) > sample_size:
    X_train_man = X_train_man.sample(n=sample_size, random_state=42)
    y_train_man = y_train_man.loc[X_train_man.index]
    X_val_man = X_val_man.sample(n=int(sample_size * 0.25), random_state=42)
    y_val_man = y_val_man.loc[X_val_man.index]

# 6. Manual Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_man)
X_val_scaled = scaler.transform(X_val_man)

print("Manual Preprocessing Complete!")
print("Preprocessed Training Shape:", X_train_scaled.shape)

Manual Preprocessing Complete!
Preprocessed Training Shape: (50000, 32)


## 5. Training Models in Individual Cells
We will now train each model individually using our manually preprocessed data.

### 5.1 Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train_man)
y_pred_lr = lr_model.predict(X_val_scaled)

print("Linear Regression Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_lr)))
print("R2:", r2_score(y_val_man, y_pred_lr))

Linear Regression Results:
RMSE: 1.7327749761786329
R2: 0.5651727573229861


### 5.2 Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train_man)
y_pred_ridge = ridge_model.predict(X_val_scaled)

print("Ridge Regression Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_ridge)))
print("R2:", r2_score(y_val_man, y_pred_ridge))

Ridge Regression Results:
RMSE: 1.7327734149069007
R2: 0.565173540902142


### 5.3 Lasso Regression

In [ ]:
from sklearn.linear_model import Lasso

lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train_scaled, y_train_man)
y_pred_lasso = lasso_model.predict(X_val_scaled)

print("Lasso Regression Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_lasso)))
print("R2:", r2_score(y_val_man, y_pred_lasso))

Lasso Regression Results:
RMSE: 1.8831827963962486
R2: 0.4864090448668583


### 5.4 Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(max_depth=7, random_state=42)
dt_model.fit(X_train_scaled, y_train_man)
y_pred_dt = dt_model.predict(X_val_scaled)

print("Decision Tree Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_dt)))
print("R2:", r2_score(y_val_man, y_pred_dt))

Decision Tree Results:
RMSE: 1.5772127514688983
R2: 0.639742550675525


### 5.5 Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train_man)
y_pred_rf = rf_model.predict(X_val_scaled)

print("Random Forest Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_rf)))
print("R2:", r2_score(y_val_man, y_pred_rf))

Random Forest Results:
RMSE: 1.1006930223000557
R2: 0.8245453683207082


### 5.6 Gradient Boosting Regressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(n_estimators=50, random_state=42)
gb_model.fit(X_train_scaled, y_train_man)
y_pred_gb = gb_model.predict(X_val_scaled)

print("Gradient Boosting Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_gb)))
print("R2:", r2_score(y_val_man, y_pred_gb))

Gradient Boosting Results:
RMSE: 1.5906744668049988
R2: 0.633566618400602


## 6. Outlier Detection and Removal
We will use the Interquartile Range (IQR) method to identify and remove outliers from our key operational numerical features (e.g., power, wind speed, generator speed).

In [ ]:
# Define key numerical columns where extreme outliers might skew the regression models
outlier_features = [
    'active_power_calculated_by_converter',
    'active_power_raw',
    'generator_speed',
    'wind_speed_raw'
]

# Keep a copy of our training data before outlier removal
X_train_no_outliers = X_train_man.copy()
y_train_no_outliers = y_train_man.copy()

print(f"Initial shape: {X_train_no_outliers.shape}")

# Apply IQR method to filter outliers
for col in outlier_features:
    if col in X_train_no_outliers.columns:
        Q1 = X_train_no_outliers[col].quantile(0.25)
        Q3 = X_train_no_outliers[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Filter both X and y
        mask = (X_train_no_outliers[col] >= lower_bound) & (X_train_no_outliers[col] <= upper_bound)
        X_train_no_outliers = X_train_no_outliers[mask]
        y_train_no_outliers = y_train_no_outliers.loc[X_train_no_outliers.index]

print(f"Shape after outlier removal: {X_train_no_outliers.shape}")

# Scale the cleaned training and original validation data
scaler_clean = StandardScaler()
X_train_scaled_clean = scaler_clean.fit_transform(X_train_no_outliers)
X_val_scaled_clean = scaler_clean.transform(X_val_man)

Initial shape: (50000, 32)
Shape after outlier removal: (47529, 32)


## 7. Retraining All Regression Algorithms (Outlier-Free)

### 7.1 Linear Regression (Retrained)

In [ ]:
lr_clean = LinearRegression()
lr_clean.fit(X_train_scaled_clean, y_train_no_outliers)
y_pred_lr_clean = lr_clean.predict(X_val_scaled_clean)

print("Linear Regression (Clean) Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_lr_clean)))
print("R2:", r2_score(y_val_man, y_pred_lr_clean))

Linear Regression (Clean) Results:
RMSE: 1.733721324119137
R2: 0.5646976693973452


### 7.2 Ridge Regression (Retrained)

In [ ]:
ridge_clean = Ridge(alpha=1.0)
ridge_clean.fit(X_train_scaled_clean, y_train_no_outliers)
y_pred_ridge_clean = ridge_clean.predict(X_val_scaled_clean)

print("Ridge Regression (Clean) Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_ridge_clean)))
print("R2:", r2_score(y_val_man, y_pred_ridge_clean))

Ridge Regression (Clean) Results:
RMSE: 1.7336981376362979
R2: 0.5647093126324905


### 7.3 Lasso Regression (Retrained)

In [ ]:
lasso_clean = Lasso(alpha=0.1)
lasso_clean.fit(X_train_scaled_clean, y_train_no_outliers)
y_pred_lasso_clean = lasso_clean.predict(X_val_scaled_clean)

print("Lasso Regression (Clean) Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_lasso_clean)))
print("R2:", r2_score(y_val_man, y_pred_lasso_clean))

Lasso Regression (Clean) Results:
RMSE: 1.8805076548009487
R2: 0.4878671642460247


### 7.4 Decision Tree Regressor (Retrained)

In [ ]:
dt_clean = DecisionTreeRegressor(max_depth=7, random_state=42)
dt_clean.fit(X_train_scaled_clean, y_train_no_outliers)
y_pred_dt_clean = dt_clean.predict(X_val_scaled_clean)

print("Decision Tree (Clean) Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_dt_clean)))
print("R2:", r2_score(y_val_man, y_pred_dt_clean))

Decision Tree (Clean) Results:
RMSE: 1.5787387612633015
R2: 0.6390450894735916


### 7.5 Random Forest Regressor (Retrained)

In [ ]:
rf_clean = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)
rf_clean.fit(X_train_scaled_clean, y_train_no_outliers)
y_pred_rf_clean = rf_clean.predict(X_val_scaled_clean)

print("Random Forest (Clean) Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_rf_clean)))
print("R2:", r2_score(y_val_man, y_pred_rf_clean))

Random Forest (Clean) Results:
RMSE: 1.1163585217357352
R2: 0.8195155482907639


### 7.6 Gradient Boosting Regressor (Retrained)

In [ ]:
gb_clean = GradientBoostingRegressor(n_estimators=50, random_state=42)
gb_clean.fit(X_train_scaled_clean, y_train_no_outliers)
y_pred_gb_clean = gb_clean.predict(X_val_scaled_clean)

print("Gradient Boosting (Clean) Results:")
print("RMSE:", np.sqrt(mean_squared_error(y_val_man, y_pred_gb_clean)))
print("R2:", r2_score(y_val_man, y_pred_gb_clean))

Gradient Boosting (Clean) Results:
RMSE: 1.6082089156701642
R2: 0.6254434971024696


## 8. Best Performing Model Extraction
Based on our comparisons, the **Random Forest Regressor** trained on the manual pipeline data with original outliers preserved performed best ($R^2 \approx 0.8245$). We will extract and finalize this model in a single cell below.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 1. Initialize the best performing model configuration
best_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

# 2. Train on the full manual scaled training set (which retains crucial high-power/high-wind operational variance)
print("Training the Best Model (Random Forest)...")
best_model.fit(X_train_scaled, y_train_man)

# 3. Predict and evaluate
y_pred_best = best_model.predict(X_val_scaled)

rmse_best = np.sqrt(mean_squared_error(y_val_man, y_pred_best))
r2_best = r2_score(y_val_man, y_pred_best)

print("\n--- Best Model Evaluation Metrics ---")
print(f"Model: Random Forest Regressor")
print(f"RMSE:  {rmse_best:.4f}")
print(f"R2:    {r2_best:.4f}")

Training the Best Model (Random Forest)...

--- Best Model Evaluation Metrics ---
Model: Random Forest Regressor
RMSE:  1.1007
R2:    0.8245


## 9. Export Changes and Model Documentation
We will now generate a markdown file summarizing the entire preprocessing workflow, the outlier experiments, and the final model selection results.

In [ ]:
markdown_content = """# Project Documentation & Summary of Changes

## 1. Overview
This project involves building a regression model to predict a target metric from wind turbine sensor and operational logs.

## 2. Data Preparation & Preprocessing Workflow
- **Deduplication:** Dropped duplicate rows from the dataset.
- **Temporal Feature Extraction:** Parsed `timestamp` to extract numeric components: `year`, `month`, `day`, and `hour` to retain temporal patterns without high-dimensionality issues.
- **Handling Missing Values:** Manually imputed numerical columns with their median values and categorical columns with their mode.
- **Categorical Encoding:** Converted the categorical identifier `turbine_id` into dummy variables using manual one-hot encoding (`pd.get_dummies`).
- **Feature Scaling:** Applied standard scaling (`StandardScaler`) to normalize the numerical variables.

## 3. Outlier Handling Experimentation
We applied the **Interquartile Range (IQR)** method to filter outliers on four highly critical, operational columns:
- `active_power_calculated_by_converter`
- `active_power_raw`
- `generator_speed`
- `wind_speed_raw`

**Result of IQR Filtering:** Outlier removal dropped the training subset size from **50,000** rows down to **47,529** rows (removing 2,471 outlier rows).

## 4. Model Evaluation & Comparison
We trained and evaluated several algorithms before and after outlier removal:

| Regression Model | R² (With Outliers / Full Variance) | R² (Outliers Removed via IQR) |
| :--- | :---: | :---: |
| **Linear Regression** | 0.5652 | 0.5647 |
| **Ridge Regression** | 0.5652 | 0.5647 |
| **Lasso Regression** | 0.4864 | 0.4879 |
| **Decision Tree** | 0.6397 | 0.6390 |
| **Random Forest** | **0.8245** | 0.8195 |
| **Gradient Boosting** | 0.6336 | 0.6254 |

### Key Takeaway:
- **Random Forest** trained on the dataset containing the outliers yielded the absolute best overall performance ($R^2 \approx 0.8245$).
- Removing outliers slightly degraded the performance of robust non-linear tree models (like Random Forest and Gradient Boosting), showing that extreme wind and power states contain important underlying operational behaviors that these trees successfully capture.

## 5. Final Model
The final selected model is a **Random Forest Regressor** (50 estimators, max depth of 12) trained on the full manually preprocessed dataset.
"""

# Write content to changes_summary.md
with open('changes_summary.md', 'w') as f:
    f.write(markdown_content)

print("Markdown file 'changes_summary.md' has been created successfully!")


Markdown file 'changes_summary.md' has been created successfully!
